## 1. Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!mkdir -p /content/drive/MyDrive/ecri_neural_network_runs
!ln -sfn /content/drive/MyDrive/ecri_neural_network_runs results

## 2. Data

In [3]:
!git clone --depth 1 https://github.com/ecri-data/ECRI-Dataset.git /content/ecri
!mkdir -p data notebooks
!cp $(find /content/ecri -name "ECRIDataSet.csv") $(find /content/ecri -name "ECRIRawData.csv") data/
!ls data

Cloning into '/content/ecri'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 17 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 1.32 MiB | 10.10 MiB/s, done.
ECRIDataSet.csv  ECRIRawData.csv


## 3. Scripts (upload the three `.py` files into `notebooks/`)

In [5]:
!ls notebooks
!python notebooks/Supriya_Neural_network_implementation.py --help | grep -q "select-on-test" && grep -q 'default="assessments"' notebooks/Supriya_Neural_network_implementation.py && echo "scripts: OK" || echo "WRONG VERSION: upload the latest scripts from the repository"

Supriya_kNN_extension.py		  Supriya_Neural_network_report.py
Supriya_Neural_network_implementation.py
scripts: OK


## 4. Quick check

In [6]:
!python notebooks/Supriya_Neural_network_implementation.py --dataset data/ECRIDataSet.csv --raw-dataset data/ECRIRawData.csv --output-dir /content/smoke --models neural --group school --target RMwidRS_target --epochs 3 --pretrain-epochs 2 --no-tsne | tail -3
!rm -rf /content/smoke

school RMwidRS_target       MaskMLP          all       0.525903                0.498762     0.563791
school RMwidRS_target       MaskMLP intervention       0.569485                0.501312     0.641909
Figures written to /content/smoke/figures


## 5. Experiments

In [7]:
%%bash
export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1
BASE="--dataset data/ECRIDataSet.csv --raw-dataset data/ECRIRawData.csv"
run() {
  name=$1; done_file=$2; shift 2
  if [ -f "results/neural_network/$name/$done_file" ]; then echo "skip $name"; return; fi
  python notebooks/Supriya_Neural_network_implementation.py $BASE --output-dir results/neural_network/$name "$@" \
    > log_${name//\//_}.txt 2>&1 && echo "done $name" || echo "FAILED $name (see log_${name//\//_}.txt)"
}
throttle() { while (( $(jobs -rp | wc -l) >= 2 )); do wait -n; done; }

throttle; run main loss_ablation.csv --seed 42 --importance --loss-ablation &
for s in 43 44 45 46;    do throttle; run seeds/s$s results.csv --seed $s --no-tsne & done
for s in 42 43 44 45 46; do throttle; run train/s$s results.csv --pretrain-scope train --seed $s --no-tsne & done
for s in 42 43 44;       do throttle; run all946/s$s results.csv --eval-rows all --seed $s --no-tsne & done
for s in 42 43 44;       do throttle; run complete/all-s$s results.csv --eval-rows complete --seed $s --no-tsne & done
for s in 42 43 44;       do throttle; run complete/train-s$s results.csv --eval-rows complete --pretrain-scope train --seed $s --no-tsne & done
for s in 42 43 44;       do throttle; run simsiam/s$s results.csv --pretrain-loss simsiam --seed $s --no-tsne & done
for s in 42 43 44;       do throttle; run contrastive/s$s results.csv --pretrain-loss contrastive --seed $s --no-tsne & done
for e in 5 10;           do throttle; run pretrain_epochs/e$e results.csv --pretrain-epochs $e --seed 42 --no-tsne & done
throttle; run diagnostic_test_selection results.csv --select-on-test --seed 42 --no-tsne &
wait
echo "ALL FINISHED"

done seeds/s43
done seeds/s44
done seeds/s45
done seeds/s46
done train/s42
done train/s43
done train/s44
done train/s45
done train/s46
done all946/s42
done all946/s43
done all946/s44
done complete/all-s42
done main
done complete/all-s43
done complete/all-s44
done complete/train-s42
done complete/train-s43
done simsiam/s42
done complete/train-s44
done simsiam/s43
done simsiam/s44
done contrastive/s42
done contrastive/s43
done contrastive/s44
done pretrain_epochs/e5
done pretrain_epochs/e10
done diagnostic_test_selection
ALL FINISHED


## 6. kNN extension

In [8]:
!python notebooks/Supriya_kNN_extension.py --raw-dataset data/ECRIRawData.csv | tail -12

                       accuracy  balanced_accuracy  pr_auc
model                                                     
LogisticRegression        0.609              0.593   0.663
XGBoost                   0.604              0.593   0.674
MLP_mean                  0.596              0.576   0.655
MLP_zero                  0.591              0.568   0.647
MLP_indicator             0.586              0.562   0.647
MaskMLP                   0.579              0.547   0.638
kNN_raw                   0.572              0.546   0.616
kNN_MaskMLP_embedding     0.515              0.504   0.575
Comparison written to results/knn_extension
Saved kNN results to results/knn_extension


## 7. Report tables

In [9]:
!python notebooks/Supriya_Neural_network_report.py --outputs results/neural_network

Runs found: all946/s42, all946/s43, all946/s44, complete/all-s42, complete/all-s43, complete/all-s44, complete/train-s42, complete/train-s43, complete/train-s44, contrastive/s42, contrastive/s43, contrastive/s44, diagnostic_test_selection, main, pretrain_epochs/e10, pretrain_epochs/e5, seeds/s43, seeds/s44, seeds/s45, seeds/s46, simsiam/s42, simsiam/s43, simsiam/s44, train/s42, train/s43, train/s44, train/s45, train/s46
Report written to results/neural_network/report (primary setting: loss=cosine, pretrain=all, rows=assessments, variant=standard)


## 8. Download

In [10]:
!zip -qr /content/neural_network_results.zip results/neural_network results/knn_extension
from google.colab import files
files.download("/content/neural_network_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>